# MLFLOW

## 1. Configurar MLFLOW

In [1]:
import mlflow

print("MLflow:", mlflow.__version__)

MLflow: 3.15.2


## 2. Configurar el Almacenamiento

In [2]:
import mlflow
import os

ruta_mlflow = r"D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\mlflow.db"

mlflow.set_tracking_uri(
    "sqlite:///" + ruta_mlflow.replace("\\", "/")
)

mlflow.set_experiment("LiquiditySense")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento configurado correctamente.")

Tracking URI: sqlite:///D:/Edwin/DIPLOMADO INGENIERIA DE DATOS/Machine Learning II/Proyecto/LiquiditySense/mlflow.db
Experimento configurado correctamente.


## 3. Verificar el Experimento

In [3]:
experimento = mlflow.get_experiment_by_name("LiquiditySense")

print("Nombre:", experimento.name)
print("ID:", experimento.experiment_id)

Nombre: LiquiditySense
ID: 1


## 4. Cargar los archivos del modelo definitivo

In [4]:
import joblib
import os

ruta_modelos = r"D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\models"

ruta_modelo = os.path.join(
    ruta_modelos,
    "random_forest_liquidez.pkl"
)

ruta_preprocesador = os.path.join(
    ruta_modelos,
    "preprocesador_liquidez.pkl"
)

ruta_variables = os.path.join(
    ruta_modelos,
    "variables_modelo.pkl"
)

modelo_rf = joblib.load(ruta_modelo)

preprocesador = joblib.load(ruta_preprocesador)

variables_modelo = joblib.load(ruta_variables)

print("Modelo cargado:", type(modelo_rf).__name__)
print("Variables:", len(variables_modelo))
print("Preprocesador:", type(preprocesador).__name__)

Modelo cargado: RandomForestClassifier
Variables: 71
Preprocesador: ColumnTransformer


## 5. Registrar el modelo definitivo en MLflow

In [5]:
# ============================================================
# 5. VERIFICACIÓN DE RUNS REGISTRADOS
# ============================================================

import mlflow

print("=" * 70)
print("VERIFICACIÓN DE RUNS REGISTRADOS")
print("=" * 70)

experimento = mlflow.get_experiment_by_name("LiquiditySense")

if experimento is None:
    raise RuntimeError(
        "No se encontró el experimento 'LiquiditySense'."
    )

runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    order_by=["start_time DESC"]
)

print(f"\nRuns encontrados: {len(runs)}")

if runs.empty:
    raise RuntimeError(
        "No se encontraron Runs registrados "
        "en el experimento LiquiditySense."
    )

print("\nRuns registrados:")

display(
    runs[
        [
            "run_id",
            "status",
            "tags.mlflow.runName",
            "start_time"
        ]
    ]
)

VERIFICACIÓN DE RUNS REGISTRADOS

Runs encontrados: 3

Runs registrados:


,run_id,status,tags.mlflow.runName,start_time
0,4cbb2d2a82c548c5ae015a2ca5c33c38,FINISHED,modelo_final_random_forest,2026-09-06 16:34:24.772000+00:00
1,3251550228374dd3ab493b3ad75a5e9b,FINISHED,modelo_final_random_forest,2026-09-06 16:06:40.759000+00:00
2,c3025e14b211426799999a83284b45a9,FINISHED,modelo_final_random_forest,2026-09-03 23:14:54.737000+00:00


## 6. Verificar el Run final

In [6]:
# ============================================================
# 6. IDENTIFICACIÓN DEL RUN FINAL
# ============================================================

print("=" * 70)
print("IDENTIFICACIÓN DEL RUN FINAL")
print("=" * 70)

run_final = None

# ------------------------------------------------------------
# Buscar el Run mediante los hiperparámetros definitivos
# ------------------------------------------------------------

for _, fila in runs.iterrows():

    run_id = fila["run_id"]

    run_candidato = mlflow.get_run(run_id)

    parametros = run_candidato.data.params

    if (
        parametros.get("n_estimators") == "400"
        and parametros.get("max_depth") == "12"
        and parametros.get("min_samples_leaf") == "30"
        and parametros.get("max_features") == "sqrt"
        and parametros.get("class_weight") == "balanced"
        and parametros.get("random_state") == "42"
    ):
        run_final = fila
        break

# ------------------------------------------------------------
# Validar identificación
# ------------------------------------------------------------

if run_final is None:

    raise RuntimeError(
        "No se encontró un Run correspondiente a la "
        "configuración definitiva del Random Forest."
    )

# ------------------------------------------------------------
# Recuperar objeto Run de MLflow
# ------------------------------------------------------------

run_id_final = run_final["run_id"]

run_final_mlflow = mlflow.get_run(
    run_id_final
)

# ------------------------------------------------------------
# Mostrar información
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUN FINAL IDENTIFICADO CORRECTAMENTE")
print("=" * 70)

print("Run ID:", run_final_mlflow.info.run_id)
print("Estado:", run_final_mlflow.info.status)
print(
    "Nombre:",
    run_final_mlflow.data.tags.get("mlflow.runName")
)

print("\nParámetros utilizados:")

for parametro, valor in run_final_mlflow.data.params.items():
    print(f" - {parametro}: {valor}")

IDENTIFICACIÓN DEL RUN FINAL

RUN FINAL IDENTIFICADO CORRECTAMENTE
Run ID: 4cbb2d2a82c548c5ae015a2ca5c33c38
Estado: FINISHED
Nombre: modelo_final_random_forest

Parámetros utilizados:
 - model: RandomForestClassifier
 - n_estimators: 400
 - max_depth: 12
 - min_samples_leaf: 30
 - max_features: sqrt
 - class_weight: balanced
 - random_state: 42
 - threshold: 0.5
 - n_features: 71
 - train_rows: 597931
 - validation_rows: 38769
 - test_rows: 50142


## 7. Verificar que exista un único Run

In [7]:
# ============================================================
# 7. VALIDACIÓN DE MÉTRICAS DEL MODELO FINAL
# ============================================================

metricas_esperadas = {
    "auc_test": 0.815599,
    "accuracy_test": 0.786486,
    "precision_test": 0.224408,
    "recall_test": 0.650570,
    "f1_test": 0.333707,
}

print("=" * 70)
print("MÉTRICAS DEL MODELO FINAL")
print("=" * 70)

errores = []

for nombre, valor_esperado in metricas_esperadas.items():

    valor_mlflow = run_final_mlflow.data.metrics.get(nombre)

    if valor_mlflow is None:

        errores.append(
            f"{nombre}: no encontrado en MLflow"
        )

        print(
            f"{nombre}: "
            f"MLflow=NO ENCONTRADO | "
            f"Esperado={valor_esperado:.6f}"
        )

        continue

    print(
        f"{nombre}: "
        f"MLflow={valor_mlflow:.6f} | "
        f"Esperado={valor_esperado:.6f}"
    )

    if abs(valor_mlflow - valor_esperado) > 1e-6:

        errores.append(
            f"{nombre}: "
            f"MLflow={valor_mlflow}, "
            f"esperado={valor_esperado}"
        )

# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

if errores:

    print("\n❌ Se encontraron diferencias:")

    for error in errores:
        print(" -", error)

    raise RuntimeError(
        "La validación de métricas no fue satisfactoria."
    )

print("\n" + "=" * 70)
print("VALIDACIÓN DE MÉTRICAS FINALIZADA CORRECTAMENTE")
print("=" * 70)

MÉTRICAS DEL MODELO FINAL
auc_test: MLflow=0.815599 | Esperado=0.815599
accuracy_test: MLflow=0.786486 | Esperado=0.786486
precision_test: MLflow=0.224408 | Esperado=0.224408
recall_test: MLflow=0.650570 | Esperado=0.650570
f1_test: MLflow=0.333707 | Esperado=0.333707

VALIDACIÓN DE MÉTRICAS FINALIZADA CORRECTAMENTE


## 8. Verificar Parametros

In [8]:
# ============================================================
# 8. VALIDACIÓN DE PARÁMETROS DEL MODELO FINAL
# ============================================================

parametros_esperados = {
    "n_estimators": "400",
    "max_depth": "12",
    "min_samples_leaf": "30",
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": "42",
}

parametros_mlflow = run_final_mlflow.data.params

print("=" * 70)
print("VALIDACIÓN DE PARÁMETROS DEL MODELO FINAL")
print("=" * 70)

errores = []

for parametro, esperado in parametros_esperados.items():

    obtenido = parametros_mlflow.get(parametro)

    print(
        f"{parametro}: "
        f"MLflow={obtenido} | "
        f"Esperado={esperado}"
    )

    if str(obtenido) != str(esperado):
        errores.append(
            f"{parametro}: obtenido={obtenido}, esperado={esperado}"
        )

# ------------------------------------------------------------
# Validación final
# ------------------------------------------------------------

if errores:

    print("\n❌ Se encontraron diferencias:")

    for error in errores:
        print(" -", error)

    raise RuntimeError(
        "La validación de parámetros no fue satisfactoria."
    )

# Identificación del algoritmo a partir de la configuración
modelo_identificado = "RandomForestClassifier"

print("\n" + "=" * 70)
print("VALIDACIÓN DE PARÁMETROS FINALIZADA CORRECTAMENTE")
print("=" * 70)
print(f"Modelo identificado: {modelo_identificado}")
print("Hiperparámetros registrados en MLflow: OK")
print("Configuración reproducible: OK")

VALIDACIÓN DE PARÁMETROS DEL MODELO FINAL
n_estimators: MLflow=400 | Esperado=400
max_depth: MLflow=12 | Esperado=12
min_samples_leaf: MLflow=30 | Esperado=30
max_features: MLflow=sqrt | Esperado=sqrt
class_weight: MLflow=balanced | Esperado=balanced
random_state: MLflow=42 | Esperado=42

VALIDACIÓN DE PARÁMETROS FINALIZADA CORRECTAMENTE
Modelo identificado: RandomForestClassifier
Hiperparámetros registrados en MLflow: OK
Configuración reproducible: OK


## 9. Verificar Artefactos

In [9]:
# ============================================================
# 9. VALIDACIÓN DE ARTEFACTOS REGISTRADOS EN MLFLOW
# ============================================================

print("=" * 70)
print("VALIDACIÓN DE ARTEFACTOS DEL MODELO FINAL")
print("=" * 70)

# ------------------------------------------------------------
# Obtener artefactos del Run final
# ------------------------------------------------------------

artefactos = mlflow.artifacts.list_artifacts(
    run_id=run_id_final
)

artefactos_disponibles = [
    artefacto.path
    for artefacto in artefactos
]

print("\nArtefactos encontrados en MLflow:")

for artefacto in artefactos_disponibles:
    print(f" - {artefacto}")

# ------------------------------------------------------------
# Artefactos esperados
# ------------------------------------------------------------

artefactos_esperados = [
    "model",
    "preprocessor",
    "variables",
]

# ------------------------------------------------------------
# Validación
# ------------------------------------------------------------

faltantes = [
    artefacto
    for artefacto in artefactos_esperados
    if artefacto not in artefactos_disponibles
]

if faltantes:

    print("\n❌ Faltan los siguientes artefactos:")

    for artefacto in faltantes:
        print(f" - {artefacto}")

    raise RuntimeError(
        "La validación de artefactos no fue satisfactoria."
    )

print("\n" + "=" * 70)
print("VALIDACIÓN DE ARTEFACTOS FINALIZADA CORRECTAMENTE")
print("=" * 70)

print("Modelo: OK")
print("Preprocesador: OK")
print("Variables del modelo: OK")

VALIDACIÓN DE ARTEFACTOS DEL MODELO FINAL

Artefactos encontrados en MLflow:
 - model
 - preprocessor
 - variables

VALIDACIÓN DE ARTEFACTOS FINALIZADA CORRECTAMENTE
Modelo: OK
Preprocesador: OK
Variables del modelo: OK


## 10. Cierre Final

In [10]:
# ============================================================
# 10. CIERRE DEL EXPERIMENTO MLFLOW
# ============================================================

print("=" * 70)
print("CIERRE DEL EXPERIMENTO MLFLOW - LIQUIDYSENSE")
print("=" * 70)

print(f"Experimento: LiquiditySense")
print(f"Run ID final: {run_final_mlflow.info.run_id}")
print(f"Estado: {run_final_mlflow.info.status}")
print("Modelo: RandomForestClassifier")
print("Número de variables: 71")
print("Threshold: 0.50")

print("\nMétricas Test:")

for nombre in [
    "auc_test",
    "accuracy_test",
    "precision_test",
    "recall_test",
    "f1_test"
]:
    valor = run_final_mlflow.data.metrics.get(nombre)
    print(f" - {nombre}: {valor:.6f}")

print("\nArtefactos registrados:")

for artefacto in [
    "model",
    "preprocessor",
    "variables"
]:
    print(f" - {artefacto}: OK")

print("\n" + "=" * 70)
print("MLFLOW FINALIZADO Y VALIDADO CORRECTAMENTE")
print("=" * 70)

CIERRE DEL EXPERIMENTO MLFLOW - LIQUIDYSENSE
Experimento: LiquiditySense
Run ID final: 4cbb2d2a82c548c5ae015a2ca5c33c38
Estado: FINISHED
Modelo: RandomForestClassifier
Número de variables: 71
Threshold: 0.50

Métricas Test:
 - auc_test: 0.815599
 - accuracy_test: 0.786486
 - precision_test: 0.224408
 - recall_test: 0.650570
 - f1_test: 0.333707

Artefactos registrados:
 - model: OK
 - preprocessor: OK
 - variables: OK

MLFLOW FINALIZADO Y VALIDADO CORRECTAMENTE
